# Uplift Modeling in Marketing — Notebook 2: Response-Targeting Baseline

> Continuation of `01_Framing_EDA_PT.ipynb`. This notebook does not share the kernel of
> the previous one — reloads below, at a low cost, exactly what is needed (raw data and
> the arm/treatment constants). No EDA results are recalculated or re-presented; that
> is already in notebook 1.

---

## Contents

- [Setup — Retaking from S1/S2](#setup)
- [Section 3 — Baseline: Response-Targeting Baseline](#s3)
    - [3.1 Preparation: Splits and Pooled Treatment](#s3-1)
    - [3.2 Response-Targeting Baseline: $P(\text{visit}=1 \mid X)$ in the treated arm](#s3-2)
    - [3.3 Evaluation of the Response-Targeting Baseline with Uplift Metrics (Validation)](#s3-3)
    - [3.4 Fast T-learner: Provisional Uplift Reference](#s3-4)
    - [3.5 Propensity vs. Uplift: Do the Two Rankings Agree?](#s3-5)
    - [3.6 Sealed Test Confirmation Rule (Written Before Sealed Test Opens)](#s3-6)
    - [Synthesis of Section 3](#s3-summary)

---

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning

# Path bootstrap: allows `from src...` from the notebooks directory.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SEED
from src.i18n import make_lang
from src.viz import apply_plot_style

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)
warnings.filterwarnings('ignore', category=FutureWarning)
# The internal causalml meta-learner propensity model (S4) uses an
# elastic-net solver that does not converge within the default max_iter on
# small samples. This does not affect the result because it is only a
# nuisance model, but it pollutes the output.
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Idioma canônico deste notebook é PT — passthrough, sem chamada de rede.
# EN edition: only this line is switched to make_lang('en').
lang = make_lang('en')

apply_plot_style()

In [ ]:
# Stack causal (verifique a instalação antes da primeira run)
# pip install econml causalml scikit-uplift shap mlflow

# Meta-learners e árvores de uplift
from causalml.inference.meta import BaseSRegressor, BaseTRegressor, BaseXRegressor, BaseRRegressor
from causalml.inference.tree import UpliftRandomForestClassifier

# Causal Forest com intervalos de confiança
from econml.dml import CausalForestDML
from econml.metalearners import TLearner, SLearner, XLearner

# Avaliação
from sklift.metrics import uplift_at_k, qini_auc_score, uplift_auc_score

from src.compat import patch_sklearn_matplotlib_support
patch_sklearn_matplotlib_support()
from sklift.viz import plot_qini_curve, plot_uplift_curve

# Interpretabilidade
import shap

In [ ]:
from src.config import ARTIFACTS_DIR, RETRAIN

ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)

# MLflow — ajuste o URI conforme seu setup Windows
# mlflow.set_tracking_uri('file:///C:/path/to/mlruns')
# mlflow.set_experiment('uplift_hillstrom')

<a id="setup"></a>

## Setup — Retaking from S1/S2

We reload here what's necessary for Section 3: the raw data and the arm/treatment constants (`ARMS`, `TREATMENT`). Nothing here has been computed before — it's the same deterministic reload from `01_Framing_EDA_PT.ipynb`.

In [ ]:
from src.config import ARMS, TREATMENT_COL as TREATMENT
from src.data import load_hillstrom

df = load_hillstrom()

<a id='s3'></a>
<a id="s3"></a>

# Section 3 — Baseline: Response-Targeting Baseline

**Objectives:**

1. Establish the benchmark that this project aims to beat: a response model $P(\text{visit}=1 \mid X)$ trained **only on the treated arm** — exactly what marketing does in practice when ranking who to treat without an explicit notion of incremental effect.
2. Rank the validation set by this probability and evaluate it with the **same uplift metrics** that will decide the winner in S6 (Qini, AUUC) — so that the comparison with the meta-learners in S4-S5 is direct.
3. Compare this ranking with a fast and untuned T-learner (provisional reference of incremental effect) via Spearman correlation. Working hypothesis: **the two rankings should disagree little** — ranking by response probability and ranking by incremental effect of treatment are, conceptually, different questions.
4. Write, **before any sealed test number exists**, the confirmation rule that will be applied in S6 (Rule Absolute #2 of this project's protocol).

**Treatment from here (S3–S6): pooled.** $T=1$ for any email arm (Mens or Womens), $T=0$ for No E-Mail — 42,667 treated vs. 21,333 controls. Individual arm separation only returns in S8, for the 3-way policy attribution.

**What DOES NOT happen in this section.** No cell below touches the sealed test set (`src/splits.py::load_sealed_test`), nor does it make `predict` on it. Only training and validation are used.

<a id='s3-1'></a>
<a id="s3-1"></a>

## 3.1 Preparation: Splits and Pooled Treatment

We now materialize the training and validation sets via `make_splits` — S3–S6 use only these two; the sealed test is stored on disk until S6. The pooled treatment (`treatment`: 1 = any email, 0 = No E-Mail) is added via `add_pooled_treatment`, without modifying the original `segment` column — it is used again in S8, for the three-way attribution policy.

In [ ]:
from src.config import POOLED_TREATMENT_COL, PRIMARY_OUTCOME
from src.data import add_pooled_treatment
from src.splits import make_splits

df_pooled = add_pooled_treatment(df)
split_idx = make_splits(df_pooled)
train_df = df_pooled.loc[split_idx['train_idx']].copy()
val_df = df_pooled.loc[split_idx['val_idx']].copy()

labels = lang({'header': 'Tamanho das partições (teste permanece selado)'})
print(f"{labels['header']}:")
print(f"  Training:   {len(train_df):>6} rows | treated: {int(train_df[POOLED_TREATMENT_COL].sum()):>6}")
print(f"  Validation: {len(val_df):>6} rows | treated: {int(val_df[POOLED_TREATMENT_COL].sum()):>6}")

In [ ]:
from src.viz import plot_split_overview

labels = lang({
    'partition_row': 'Partição',
    'arm_row': 'Braço',
    'outcome_row': 'Outcome (visit)',
    'train': 'Treino',
    'val': 'Validação',
    'sealed': 'Teste selado (oculto até S6)',
    'outcome_neg': 'Não visitou (0)',
    'outcome_pos': 'Visitou (1)',
    'title': 'Composição do split',
    'subtitle': 'Treino e validação preservam a mesma proporção de braço e outcome; teste segue selado',
})
n_test = len(df_pooled) - len(train_df) - len(val_df)
fig, ax = plot_split_overview(
    train_df, val_df, n_test, TREATMENT, ARMS, PRIMARY_OUTCOME, labels,
    title=labels['title'], subtitle=labels['subtitle'],
)
plt.show()

**Reading the Graph.** Training and validation are ordered by (arm, outcome) solely to make visible, in blocks, the relative proportions of each combination – the order has no temporal significance (the dataset is not a time series). The grey block of the sealed test appears only in size: no composition of arm or outcome is revealed until Section 6, by discipline of the protocol. The visual similarity between the training and validation blocks is the graphical portrait of the stratification (arm × visit) already verified by automated testing in `tests/test_suite.py`.

<a id='s3-2'></a>
<a id="s3-2"></a>

## 3.2 Response-Targeting Baseline: $P(\text{visit}=1 \mid X)$ in the treated arm

The baseline trains a response classifier **only with treated rows** — it never sees the control arm. It answers "who, among the treated, has the highest probability of visiting the site?", which is the question that uplift modeling answers in practice marketing. It is not a causal question: nothing in this model separates "would visit anyway" from "visited because of the email".

We use LightGBM — the same class of model that will anchor the four meta-learners of S4. The choice is deliberate: if the baseline loses to the meta-learners, we want the difference to come from the **targeting strategy** (propensity vs. uplift), not the model's capability.

In [ ]:
from src.learners import fit_propensity_baseline, predict_propensity_score

propensity_model = fit_propensity_baseline(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME)
val_df['propensity_score'] = predict_propensity_score(propensity_model, val_df)
val_df[['propensity_score']].describe()

<a id='s3-3'></a>
<a id="s3-3"></a>

## 3.3 Evaluation of the Response-Targeting Baseline with Uplift Metrics (Validation)

The baseline does not estimate incremental effect — but to compare with the models of S4-S5 on an equal footing, we evaluate its **ranking** with the same metrics that will decide the winner in S6: Qini AUC, uplift AUC (AUUC), and uplift@30%. This is possible because these metrics do not require the score to be a CATE — they only require a ranking; a high propensity score is treated, for the purpose of the metric, as "high treatment priority".

In [ ]:
from src.evaluation import evaluate_ranking

baseline_metrics = evaluate_ranking(
    val_df[PRIMARY_OUTCOME].values, val_df['propensity_score'].values, val_df[POOLED_TREATMENT_COL].values,
)
print(baseline_metrics)

In [ ]:
from src.viz import add_chart_footer, add_chart_header

labels = lang({
    'title': 'Curva Qini — baseline de propensão (validação)',
    'series_name': 'Baseline de propensão',
    'subtitle': 'Baseline supera a linha aleatória, mas com margem modesta (Qini AUC = 0.0395)',
})
qini_disp = plot_qini_curve(
    val_df[PRIMARY_OUTCOME].values, val_df['propensity_score'].values, val_df[POOLED_TREATMENT_COL].values,
    perfect=True, name=labels['series_name'],
)
add_chart_header(qini_disp.figure_, title=labels['title'], subtitle=labels['subtitle'])
add_chart_footer(qini_disp.figure_, text='Source: validation, 12,800 rows | Method: LightGBM on the treated arm')
qini_disp.figure_.subplots_adjust(top=0.78, bottom=0.15)
plt.show()

**Reading.** In validation (12,800 lines, 8,538 treated), the response-targeting baseline achieves Qini AUC = 0.0395, uplift AUC (AUUC) = 0.0235, and uplift@30% = +0.0923 — that is, treating the top 30% of the ranking by propensity, the visitation rate between treated and control in this subset differs by ~9.2pp more than what would be observed by treating randomly. These three numbers are the benchmark that the meta-learners of S4 and the direct models of S5 need to beat. In this section, they are not yet formally comparable — this, with confidence intervals, is the work of S6.

<a id='s3-4'></a>
<a id="s3-4"></a>

## 3.4 Fast T-learner: Provisional Uplift Reference

To check if the propensity ranking agrees with an incremental effect ranking, we need *some* CATE estimator — even a provisional one. We adjust a simple T-learner (two independent LightGBM, one per arm, without tuning) just for this check. The definitive version, tuned and formally compared to other meta-learners, is S4's work — this model is not part of that comparison.

In [ ]:
from src.learners import fit_t_learner_quick, predict_t_learner_uplift

t_learner_models = fit_t_learner_quick(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME)
val_df['t_learner_uplift'] = predict_t_learner_uplift(t_learner_models, val_df)
val_df[['t_learner_uplift']].describe()

<a id='s3-5'></a>
<a id="s3-5"></a>

## 3.5 Propensity vs. Uplift: Do the Two Rankings Agree?

If the propensity ranking and the uplift ranking (T-learner) were essentially the same thing, there would be no reason for this project to exist — we could simply continue ranking by probability of response. The Spearman correlation between the two scores directly measures this overlap.

In [ ]:
from src.evaluation import spearman_ranking_correlation

rank_corr = spearman_ranking_correlation(val_df['propensity_score'], val_df['t_learner_uplift'])
print(rank_corr)

**Reading.** The Spearman correlation between the propensity score and the uplift of the T-learner is $\rho = 0.444$ ($p \approx 0$, which does not surprise with $n = 12.800$). This is **higher than the anticipated working hypothesis** — the two rankings are not independent, and ranking by propensity indeed carries some signal about incremental effect (plausible: in this dataset, clients with a higher basal probability of visiting also tend to respond more to the stimulus). Still, $\rho = 0.444$ is far from indicating that the two rankings are interchangeable — more than half of the ranking variation is not shared between the two approaches, and it is exactly in this divergence that uplift modeling can capture value that propensity modeling does not capture.

Note that the fast T-learner (without tuning) produced estimated uplift with a long negative tail (minimum -0.58). We do not interpret this as evidence of Sleeping Dogs in this section — it is a provisional and known noisy estimator; heterogeneity analysis is left for S7.

<a id='s3-6'></a>
<a id="s3-6"></a>

## 3.6 Sealed Test Confirmation Rule (Written Before Sealed Test Opens)

We now fix, before any sealed test number exists, the rule that will be applied in S6 to decide whether the validation winner model is confirmed in the sealed test.

> ### Sealed Test Confirmation Rule for S6
> The validation winner model (highest Qini AUC between S3–S5) is **confirmed** in the sealed test if, and only if, all the conditions below hold:
> 1. Its Qini in the sealed test is **positive**;
> 2. The 95% bootstrap confidence interval of the Qini in the sealed test **does not include zero**;
> 3. Its Qini in the sealed test **exceeds** the response-targeting baseline Qini (Section 3.3), measured in the same test.
>
> If any condition fails, the result is **"not confirmed"** — and will be reported exactly as is, without protocol adjustment (Absolute Rule #6: negative result is result).

This rule will not be revised in S6, regardless of what the validation results (S4-S5) suggest.

<a id="s3-summary"></a>

### Synthesis of Section 3

| Item | Result |
|---|---|
| Response-targeting baseline — Qini AUC (validation) | 0.0395 |
| Response-targeting baseline — uplift@30% (validation) | +9.2pp |
| Correlation (Spearman) propensity × T-learner fast | 0.444 ($p \approx 0$) |
| Confirmation rule for S6 | Written in Section 3.6, before any test result |

The response-targeting baseline is set as the benchmark to beat. The moderate correlation with the T-learner confirms that propensity and uplift are related but not equivalent signals — opening up real space for the meta-learners of S4 to surpass this baseline in uplift metrics, without being guaranteed.

**Next:** [Section 4 — Meta-learners (S/T/X/R)](#s4), with LightGBM as common base learner and the hypothesis of structural advantage of X-learner under imbalanced arms to be tested.